In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
plt.hist(df["Delivery_Time"])
plt.show()


In [ ]:
df = df.drop(columns=["Order_ID"])

In [ ]:
df.isnull().sum()

In [ ]:
for col in df.select_dtypes(include=["float64", "int64"]).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
df = df.drop_duplicates()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

X = scaler.fit_transform(X)

In [ ]:
X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(n_estimators=300, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    scores.append(mean_absolute_error(y_test, y_pred))

sum(scores) / len(scores)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

feature_importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
feature_importance.head(20).plot(kind="bar")
plt.show()

In [ ]:
plt.hist(y_pred, bins=30)
plt.show()

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf = RandomForestRegressor(n_estimators=300, random_state=42)
    cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=8, loss_function="MAE", verbose=False, random_seed=42)

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    pred_rf = rf.predict(X_test)
    pred_cb = cb.predict(X_test)

    pred_avg = (pred_rf + pred_cb) / 2
    scores.append(mean_absolute_error(y_test, pred_avg))

sum(scores) / len(scores)
